# 4_results_tto — charts and tables of a Trinidad and Tobago run

Port of Jamaica's `4_results_beta16.ipynb` (2026-10-25, step 9 of the Trinidad build): the fan chart with the
dispersion bands and the breadth strip, the watchlist chart (one panel per cluster at its own vintage), and the
accuracy table by vintage. Melissa lines removed; the shock labels carry COVID only; the clusters are the frozen groups
of `config.py` section 3 / 7. Run standalone (reads the latest `output/<date>_tto_prod`, or `TTO_LAUNCH_DATE_STR` +
`TTO_RUN_TAG`) or from `run_tto.py`, which injects `PATH_OUTPUT` and `date_str`.

In [ ]:
# [tto:results-setup]
import numpy as np, pandas as pd, os, glob, sys
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
if "date_str" not in globals() or "PATH_OUTPUT" not in globals():
    _ds = os.environ.get("TTO_LAUNCH_DATE_STR"); _tag = os.environ.get("TTO_RUN_TAG", "prod")
    _root = os.path.join(os.getcwd(), "..", "output")
    if _ds:
        OUT_SUBFOLDER = f"{_ds}_tto_{_tag}"
    else:
        _cands = sorted(f for f in os.listdir(_root) if f.endswith(f"_tto_{_tag}"))
        OUT_SUBFOLDER = _cands[-1]
    date_str = OUT_SUBFOLDER.split("_")[0]
    PATH_OUTPUT = os.path.join(_root, OUT_SUBFOLDER)
print(f"[results] date_str={date_str} PATH_OUTPUT={PATH_OUTPUT}")
COUNTRY = "Trinidad and Tobago"
SHOCK_LABELS = {"2020-06-01": "COVID"}        # display names only; the regime comes from the log

def _qlabel(d):
    d = pd.Timestamp(d); return f"{d.year}-Q{(d.month - 1) // 3 + 1}"
def quarter_axis(ax, dates, rotation=0, fontsize=9, every=None):
    """Tick the x-axis at the quarters in `dates`, labelled YYYY-Qn; thins the labels when the axis is narrow."""
    ds = list(pd.to_datetime(pd.Index(dates)).unique()); ds.sort()
    if every is None:
        fig = ax.get_figure(); width_in = ax.get_position().width * fig.get_size_inches()[0]
        per = width_in / max(len(ds), 1); every = 1 if per >= 0.7 else (2 if per >= 0.35 else 3)
    keep = list(range(0, len(ds), every)); labels = [_qlabel(ds[k]) for k in keep]
    if ax.get_xlim()[1] > 5000: ax.set_xticks([ds[k] for k in keep])
    else: ax.set_xticks(keep)
    ax.set_xticklabels(labels, rotation=rotation, ha="right" if rotation else "center", fontsize=fontsize)
def shock_marks(rl):
    """(date, label) for every quarter the automatic regime flags, with the first vintage at which it switched on."""
    first = rl[rl["is_shock"]].sort_values(["date", "Vintage"]).groupby("date")["Vintage"].first()
    return [(d, f"{SHOCK_LABELS.get(d.strftime('%Y-%m-%d'), _qlabel(d))} (shock regime from V{int(v):+d})") for d, v in first.items()]


In [ ]:
# [tto:results-fan]
# Fan chart: actual GDP, the regime-aware nowcast at V+6 (the last reading), the 80 % and 95 % bands, burn-in hatched, the live quarter
# as a dotted continuation, and the breadth strip underneath (the count behind the corroboration rule).
ra = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])
ci = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt dispersion_bands.xlsx", parse_dates=["date"])
rl = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="regime_log", parse_dates=["date"])
V_HEADLINE = 6; BURN_IN_MIN_PRINTS = 3
ra_v = ra[ra["Vintage"] == V_HEADLINE].sort_values("date").reset_index(drop=True)
ci_v = ci[ci["Vintage"] == V_HEADLINE].sort_values("date").reset_index(drop=True)
_bi = rl[rl["Vintage"] == V_HEADLINE].set_index("date")["burn_in"]
ci_v["burn_in"] = [bool(_bi.get(d, False)) for d in ci_v["date"]]
n_burn = int(ci_v["burn_in"].sum())
seg_burn = ci_v.iloc[:n_burn + 1] if n_burn else ci_v.iloc[:0]; seg_ok = ci_v.iloc[n_burn:]

fig, (ax, axb) = plt.subplots(2, 1, figsize=(13, 7.2), sharex=True, gridspec_kw={"height_ratios": [4.2, 1.1], "hspace": 0.06})
ax.fill_between(seg_ok["date"], seg_ok["lo_95"], seg_ok["hi_95"], alpha=0.12, color="tab:blue", label="95% band: full range of the pool")
ax.fill_between(seg_ok["date"], seg_ok["lo_80"], seg_ok["hi_80"], alpha=0.38, color="tab:blue", label="80% band")
if n_burn:
    ax.fill_between(seg_burn["date"], seg_burn["lo_95"], seg_burn["hi_95"], facecolor="none", edgecolor="tab:blue", hatch="//", linewidth=0.6, alpha=0.5,
                    label=f"Burn-in: weights rest on fewer than {BURN_IN_MIN_PRINTS} published prints")
    ax.fill_between(seg_burn["date"], seg_burn["lo_80"], seg_burn["hi_80"], facecolor="none", edgecolor="tab:blue", hatch="xx", linewidth=0.6, alpha=0.5)
    ax.plot(ra_v["date"].iloc[:n_burn + 1], ra_v["regime_aware_nowcast"].iloc[:n_burn + 1], color="tab:blue", linewidth=2.0, linestyle="--")
ax.plot(ra_v["date"].iloc[n_burn:], ra_v["regime_aware_nowcast"].iloc[n_burn:], color="tab:blue", linewidth=2.0, label=f"Regime-aware nowcast (V+{V_HEADLINE})")
ax.plot(ra_v["date"], ra_v["actual"], color="black", linewidth=1.5, marker="o", markersize=5, label="Actual GDP QoQ % (SA)")
for dts, lab in shock_marks(rl):
    if dts in set(ra_v["date"]):
        ax.axvline(dts, color="red", linestyle=":", linewidth=1, alpha=0.7); axb.axvline(dts, color="red", linestyle=":", linewidth=1, alpha=0.7)
        ax.text(dts, ax.get_ylim()[1] * 0.92, " " + lab, color="red", fontsize=9, va="top")
_live_path = f"{PATH_OUTPUT}/{date_str} 4_rslt live_quarter.xlsx"
lives = pd.read_excel(_live_path, sheet_name="nowcast", parse_dates=["date"]).sort_values("date") if os.path.exists(_live_path) else pd.DataFrame()
live = lives.iloc[-1] if len(lives) else None            # the latest live quarter (used by the strip below)
_x0, _y0 = ra_v["date"].iloc[-1], ra_v["regime_aware_nowcast"].iloc[-1]; _c = ci_v.iloc[-1]
_prev = {"date": _x0, "regime_aware_nowcast": _y0, "lo_95": _c["lo_95"], "hi_95": _c["hi_95"], "lo_80": _c["lo_80"], "hi_80": _c["hi_80"]}
for _, lv in lives.iterrows():                           # 2026-10-26: every unprinted quarter, each read at its own vintage
    _cx = [_prev["date"], lv["date"]]
    ax.plot(_cx, [_prev["regime_aware_nowcast"], lv["regime_aware_nowcast"]], color="tab:blue", linewidth=2.0, linestyle=":", marker="o", markevery=[1], markerfacecolor="white", markersize=7,
            label=f"Live {lv['quarter']} read at V{int(lv['Vintage']):+d} (data to {pd.Timestamp(lv['data_cut']):%b %Y}): {lv['regime_aware_nowcast']:+.1f}")
    ax.fill_between(_cx, [_prev["lo_95"], lv["lo_95"]], [_prev["hi_95"], lv["hi_95"]], facecolor="tab:blue", alpha=0.08, edgecolor="tab:blue", linestyle=":", linewidth=1.0, hatch="..")
    ax.fill_between(_cx, [_prev["lo_80"], lv["lo_80"]], [_prev["hi_80"], lv["hi_80"]], facecolor="tab:blue", alpha=0.25, edgecolor="tab:blue", linestyle=":", linewidth=1.0, hatch="..")
    ax.annotate(f"live, V{int(lv['Vintage']):+d}", xy=(lv["date"], lv["hi_95"]), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=8.5, color="tab:blue")
    if bool(lv["is_shock"]):
        ax.axvline(lv["date"], color="red", linestyle=":", linewidth=1, alpha=0.7); ax.text(lv["date"], ax.get_ylim()[1] * 0.92, f" {lv['quarter']} (shock regime, live)", color="red", fontsize=9, va="top")
    _prev = lv
ax.axhline(0, color="gray", linewidth=0.7, alpha=0.6); ax.set_ylabel("QoQ growth, %")
ax.set_title(f"{COUNTRY} GDP: regime-aware nowcast and dispersion bands (V+{V_HEADLINE})"); ax.legend(loc="lower left", fontsize=8.5); ax.grid(alpha=0.3)
rl_v = rl[rl["Vintage"] == V_HEADLINE].set_index("date").reindex(ra_v["date"])
thr, n_panel = int(rl["threshold"].iloc[0]), int(rl["n_panel"].iloc[0])
colors = np.where(rl_v["is_shock"].fillna(False).values, "red", "tab:blue")
axb.bar(rl_v.index, rl_v["breadth"].values, width=55, color=colors, alpha=0.75)
for _, lv in lives.iterrows():
    axb.bar([lv["date"]], [lv["breadth"]], width=55, color=("red" if bool(lv["is_shock"]) else "tab:blue"), alpha=0.75, hatch="//", edgecolor="white")
axb.axhline(max(3, thr // 3), color="gray", linestyle="--", linewidth=0.9, label=f"Minimum behind a watchlist alert ({max(3, thr // 3)} of {n_panel}); breadth alone does not switch the regime")
axb.set_ylabel("Broken\nvariables", fontsize=9); axb.set_ylim(0, max(thr + 2, rl_v["breadth"].max() + 2)); axb.set_xlabel("Quarter")
axb.legend(loc="upper left", fontsize=8); axb.grid(alpha=0.3)
quarter_axis(axb, list(ra_v["date"]) + (list(lives["date"]) if len(lives) else []))
plt.tight_layout()
out = f"{PATH_OUTPUT}/{date_str} 5_chart fan_headline_v{V_HEADLINE}.png"; plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
print(f"[OK] {out}  (burn-in quarters: {n_burn})")


In [ ]:
# [tto:results-watchlist]
# Watchlist chart: one panel per cluster at the vintage it is read at (frozen groups, config section 3 / 7).
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
rl = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="regime_log", parse_dates=["date"])
ra = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])
ws = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt watchlist.xlsx", sheet_name="watchlist_summary", parse_dates=["date"])
_live_path = f"{PATH_OUTPUT}/{date_str} 4_rslt live_quarter.xlsx"
lives = pd.read_excel(_live_path, sheet_name="nowcast", parse_dates=["date"]).sort_values("date") if os.path.exists(_live_path) else pd.DataFrame()
live = lives.iloc[-1] if len(lives) else None
live_cl_all = pd.read_excel(_live_path, sheet_name="clusters", parse_dates=["date"]) if live is not None else None
ALERT_PP = 4.0
PANELS = [  # (cluster, vintage, title, colour)
    ("early_vm1",   -1, r"$\bf{Early}$: frozen SHAP group of 20, raw designs x linear, read at V-1",             "tab:orange"),
    ("zero_v0",      0, r"$\bf{Zero}$: frozen iBMA group of 6, raw designs x linear, read at V0",                "tab:red"),
    ("support_vp1",  1, r"$\bf{Support}$: frozen SHAP group of 6, factor designs x ridge, read at V+1",          "tab:purple"),
    ("confirm_vp2",  2, r"$\bf{Confirmation}$: frozen SHAP group of 6, factor designs x linear, read at V+2",    "tab:green"),
    ("late_vp4",     4, r"$\bf{Late}$: frozen SHAP group of 6, factor designs x penalised, read at V+4",          "tab:brown"),
]
PANELS = [p for p in PANELS if (ws["cluster"] == p[0]).any()]   # 2026-10-26: panels only for clusters present in the run
ref = ws[ws["cluster"] == "reference"]
_nr = (len(PANELS) + 1) // 2
fig, axes = plt.subplots(_nr, 2, figsize=(13, 4 * _nr), sharex=True, sharey=True, squeeze=False)
for ax in axes.flat[len(PANELS):]: ax.set_visible(False)
for ax, (cl, V, title, col) in zip(axes.flat, PANELS):
    s = ws[(ws["cluster"] == cl) & (ws["Vintage"] == V)].set_index("date").sort_index()
    if s.empty:
        ax.set_title(title + "  [no data]", fontsize=10); continue
    ax.fill_between(s.index, s["pooled_median"] - ALERT_PP, s["pooled_median"] + ALERT_PP, color="tab:blue", alpha=0.10)
    ax.plot(s.index, s["pooled_median"], color="tab:blue", linewidth=1.2, linestyle="--")
    ax.plot(s.index, s["actual"], color="black", linewidth=1.6, marker="o", markersize=4)
    r = ra[ra["Vintage"] == V].set_index("date").sort_index()["regime_aware_nowcast"].reindex(s.index)
    ax.plot(r.index, r.values, color="tab:blue", linewidth=1.8, marker="x", markersize=5)
    rf = ref[ref["Vintage"] == V].set_index("date").sort_index()["median"].reindex(s.index)
    ax.plot(rf.index, rf.values, color="tab:gray", linewidth=1.2, linestyle="-.")
    if live is not None:
        _px, _pm0, _pcl, _pra = s.index[-1], s["pooled_median"].iloc[-1], s["median"].iloc[-1], r.values[-1]
        for _, lv in lives.iterrows():                   # 2026-10-26: every live quarter, chained in date order
            _lcs = live_cl_all[(live_cl_all["cluster"] == cl) & (live_cl_all["date"] == lv["date"])]
            if _lcs.empty: continue
            _lc = _lcs.iloc[0]; _lx = lv["date"]; _pm1 = _lc["pooled_median"]
            ax.fill_between([_px, _lx], [_pm0 - ALERT_PP, _pm1 - ALERT_PP], [_pm0 + ALERT_PP, _pm1 + ALERT_PP], facecolor="tab:blue", alpha=0.07, edgecolor="tab:blue", linestyle=":", linewidth=0.8, hatch="..")
            ax.annotate(f"live, V{int(lv['Vintage']):+d}", xy=(_lx, _pm1 + ALERT_PP), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=8, color="tab:blue")
            ax.plot([_px, _lx], [_pcl, _lc["median"]], color=col, linewidth=2.2, linestyle=":", marker="s", markevery=[1], markerfacecolor="white", markersize=5)
            ax.plot([_px, _lx], [_pm0, _pm1], color="tab:blue", linewidth=1.2, linestyle=":")
            ax.plot([_px, _lx], [_pra, lv["regime_aware_nowcast"]], color="tab:blue", linewidth=1.8, linestyle=":", marker="x", markevery=[1], markersize=5)
            if int(_lc["alert"]) == 1: ax.scatter([_lx], [_lc["median"]], s=170, facecolors="none", edgecolors="red", linewidths=1.6, zorder=5)
            _px, _pm0, _pcl, _pra = _lx, _pm1, _lc["median"], lv["regime_aware_nowcast"]
    ax.plot(s.index, s["median"], color=col, linewidth=2.2, marker="s", markersize=4)
    idx = s.index[s["alert"] == 1]
    if len(idx): ax.scatter(idx, s.loc[idx, "median"], s=170, facecolors="none", edgecolors="red", linewidths=1.6, zorder=5)
    idx_up = s.index[s["flag"] == 1] if "flag" in s.columns else []
    if len(idx_up): ax.scatter(idx_up, s.loc[idx_up, "median"], s=150, marker="^", facecolors="none", edgecolors="darkorange", linewidths=1.6, zorder=5)
    for dts, lab in shock_marks(rl):
        if dts in s.index: ax.axvline(dts, color="red", linestyle=":", linewidth=1, alpha=0.7)
    ax.axhline(0, color="gray", linewidth=0.7, alpha=0.6); ax.set_title(title, fontsize=10); ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("Quarter"); quarter_axis(ax, sorted(ws["date"].unique()) + (list(lives["date"]) if len(lives) else []))
for ax in axes[:, 0]: ax.set_ylabel("QoQ growth, %")
marks = shock_marks(rl)
handles = [Line2D([], [], color="black", marker="o", markersize=4, label="Actual GDP (SA)"),
           Line2D([], [], color="tab:blue", linewidth=1.8, marker="x", markersize=5, label="Regime-aware nowcast at that vintage"),
           Line2D([], [], color="tab:blue", linestyle="--", label="Pooled median at that vintage (alert reference)"),
           Patch(facecolor="tab:blue", alpha=0.10, label=f"+/-{ALERT_PP:.0f} pp corridor: inside = no alert"),
           Line2D([], [], color="dimgray", marker="s", markersize=4, linewidth=2.2, label="Cluster median (colour per panel)"),
           Line2D([], [], color="tab:gray", linestyle="-.", label="Reference: activity economist group"),
           Line2D([], [], marker="o", markersize=11, markerfacecolor="none", markeredgecolor="red", linestyle="none", label="Alert: cluster below the corridor (can switch the regime)"),
           Line2D([], [], marker="^", markersize=10, markerfacecolor="none", markeredgecolor="darkorange", linestyle="none", label="Flag: cluster above the corridor (information only)"),
           Line2D([], [], color="red", linestyle=":", label="Shock quarter (regime log)" + (": " + "; ".join(l for _, l in marks) if marks else ""))]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8.5, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"{COUNTRY} watchlist: the clusters against the pooled median (alert = divergence <= -{ALERT_PP:.0f} pp; flag = >= +{ALERT_PP:.0f} pp)", fontsize=13, fontweight="bold", y=0.945)
_notes = ["Clusters identified on COVID 2020-Q2 and frozen; an alert backed by at least the minimum of broken indicators switches the regime; breadth alone never does.",
          "Early: a warning before the quarter ends (V-1). Zero: at quarter end. Support: one month after. Confirmation: two months after quarter end. Late: four months after, two before the print."]
if live is not None:
    _notes.append("Live quarters (hollow markers, dotted): " + "; ".join(f"{lv['quarter']} read at V{int(lv['Vintage']):+d}" for _, lv in lives.iterrows()) + " (data cut " + f"{pd.Timestamp(live['data_cut']):%b %Y}" + "); only the cluster defined at that vintage is an alert candidate.")
fig.text(0.01, -0.045, "\n".join(_notes), ha="left", va="top", fontsize=8.5, linespacing=1.4)
plt.tight_layout(rect=(0, 0.02, 1, 0.965))
out = f"{PATH_OUTPUT}/{date_str} 5_chart watchlist_divergence.png"; plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
print(f"[OK] {out}")


In [ ]:
# [tto:results-accuracy]
# Accuracy by vintage on the test window: regime-aware nowcast, pooled median (spread sheet p50, after the pool filter
# and the reading guard), no-change forecast; band coverage; the same restricted to the quarters after the burn-in.
ra = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="nowcast", parse_dates=["date"])
rl = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt regime_aware_nowcast.xlsx", sheet_name="regime_log", parse_dates=["date"])
ci = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt dispersion_bands.xlsx", parse_dates=["date"])
sp = pd.read_excel(f"{PATH_OUTPUT}/{date_str} 4_rslt watchlist.xlsx", sheet_name="spread", parse_dates=["date"])
t = ra.merge(sp[["date", "Vintage", "p50"]], on=["date", "Vintage"]).merge(rl[["date", "Vintage", "burn_in"]], on=["date", "Vintage"]) \
      .merge(ci[["date", "Vintage", "lo_80", "hi_80", "lo_95", "hi_95"]], on=["date", "Vintage"])
act = ra.groupby("date")["actual"].first().sort_index(); t["nochange"] = t["date"].map(act.shift(1))
t["in80"] = (t["actual"] >= t["lo_80"]) & (t["actual"] <= t["hi_80"]); t["in95"] = (t["actual"] >= t["lo_95"]) & (t["actual"] <= t["hi_95"])
def _acc(g):
    e = lambda c: g[c] - g["actual"]
    return pd.Series({"quarters": len(g), "rmse_regime_aware": np.sqrt((e("regime_aware_nowcast") ** 2).mean()), "mae_regime_aware": e("regime_aware_nowcast").abs().mean(),
                      "rmse_pooled_median": np.sqrt((e("p50") ** 2).mean()), "rmse_no_change": np.sqrt((e("nochange") ** 2).dropna().mean()),
                      "in_80_band": int(g["in80"].sum()), "in_95_band": int(g["in95"].sum())})
acc_all = t.groupby("Vintage").apply(_acc).round(2); acc_ok = t[~t["burn_in"]].groupby("Vintage").apply(_acc).round(2)
sd = float(act.std()); print(f"[accuracy] test window {_qlabel(act.index.min())} to {_qlabel(act.index.max())}: actual mean {act.mean():+.2f}, sd {sd:.2f}")
print("all quarters:"); print(acc_all.to_string()); print("after burn-in:"); print(acc_ok.to_string())
out = f"{PATH_OUTPUT}/{date_str} 5_tab accuracy_by_vintage.xlsx"
with pd.ExcelWriter(out) as xw:
    acc_all.to_excel(xw, sheet_name="all_quarters"); acc_ok.to_excel(xw, sheet_name="after_burn_in"); t.round(3).to_excel(xw, sheet_name="by_quarter", index=False)
print(f"[OK] {out}")
